# PHM08 - Remaining Useful Life (RUL) Prediction
Random Forest baseline vs XGBoost, trained on the PHM08 dataset.

## 1. Setup

In [ ]:
import sys
sys.path.append("src")

!pip install -q xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

from data.preprocessing import prepare_train, select_sensors
from data.windowing import make_windows

## 2. Data preparation

In [ ]:
df, stats = prepare_train("data/train.txt")
sensors = select_sensors()

print(df.shape)
print(df["regime"].value_counts())
print(df[["RUL", "RUL_cap"]].describe())

X, y, groups = make_windows(df, sensors)
print(X.shape, y.shape, groups.nunique(), "moteurs")

## 3. Train / validation split

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

print("train:", X_train.shape, "| val:", X_val.shape)
print("moteurs train:", groups.iloc[train_idx].nunique(), "| moteurs val:", groups.iloc[val_idx].nunique())

## 4. Random Forest baseline

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_val)
rmse_rf = np.sqrt(mean_squared_error(y_val, pred_rf))
mae_rf = mean_absolute_error(y_val, pred_rf)

print("Random Forest — RMSE:", round(rmse_rf, 2), "| MAE:", round(mae_rf, 2))

## 5. XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_val)
pred_xgb_clipped = np.clip(pred_xgb, 0, 125)

rmse_xgb = np.sqrt(mean_squared_error(y_val, pred_xgb_clipped))
mae_xgb = mean_absolute_error(y_val, pred_xgb_clipped)

print("XGBoost (clippé) — RMSE:", round(rmse_xgb, 2), "| MAE:", round(mae_xgb, 2))

## 6. Evaluation / results

In [ ]:
r2_rf = r2_score(y_val, pred_rf)
r2_xgb = r2_score(y_val, pred_xgb_clipped)

print("Random Forest — RMSE:", round(rmse_rf, 2), "| MAE:", round(mae_rf, 2), "| R2:", round(r2_rf, 3))
print("XGBoost       — RMSE:", round(rmse_xgb, 2), "| MAE:", round(mae_xgb, 2), "| R2:", round(r2_xgb, 3))

## 7. Feature importance & plots

In [ ]:
importances_rf = pd.Series(rf.feature_importances_, index=X_train.columns)
print("Top features — Random Forest")
print(importances_rf.sort_values(ascending=False).head(15))

importances_xgb = pd.Series(xgb.feature_importances_, index=X_train.columns)
print("\nTop features — XGBoost")
print(importances_xgb.sort_values(ascending=False).head(15))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].scatter(y_val, pred_rf, alpha=0.3, s=10)
axes[0].plot([0, 125], [0, 125], color="red", linestyle="--")
axes[0].set_xlabel("RUL réel")
axes[0].set_ylabel("RUL prédit")
axes[0].set_title("Random Forest")

axes[1].scatter(y_val, pred_xgb_clipped, alpha=0.3, s=10)
axes[1].plot([0, 125], [0, 125], color="red", linestyle="--")
axes[1].set_xlabel("RUL réel")
axes[1].set_ylabel("RUL prédit")
axes[1].set_title("XGBoost")

plt.tight_layout()
plt.show()

## 8. Save model

In [ ]:
import os
os.makedirs("models", exist_ok=True)
joblib.dump(xgb, "models/xgb_rul_model.pkl")
print("Modèle sauvegardé : models/xgb_rul_model.pkl")